# Experiment 3: Car Evaluation with Sigmoid Activation

- Task requirement: `Use sigmoid instead of ReLU`
- Dataset: `UCI Car Evaluation`
- This notebook runs the full training script and displays the generated figures.

- This notebook can be uploaded by itself. It will recreate the required script and test files inside Colab before running.

In [ ]:
# Auto-generated setup cell for Colab-only upload workflow.
from pathlib import Path
import subprocess
import sys

EXPERIMENT_DIR = Path("/content/experiment_3_seed99")
FILES = {
  "run_car_mlp.py": "\"\"\"Experiment 3 entry point.\"\"\"\n\nfrom pathlib import Path\n\nfrom experiment_support import build_experiment_profile, expected_output_files, run_experiment, validate_config\n\n\nEXPERIMENT_CONFIG = {\n    \"experiment_name\": \"experiment_3_seed99\",\n    \"task_requirement\": \"Use sigmoid instead of ReLU\",\n    \"dataset_name\": \"car_evaluation\",\n    \"random_seed\": 99,\n    \"test_size\": 0.30,\n    \"validation_size\": 0.20,\n    \"batch_size\": 64,\n    \"activation\": \"sigmoid\",\n    \"force_single_layer\": False,\n    \"hyperparameter_grid\": [\n        {\"hidden_neurons\": [40], \"learning_rate\": 0.020, \"epochs\": 180, \"dropout_rate\": 0.0},\n        {\"hidden_neurons\": [64, 32], \"learning_rate\": 0.012, \"epochs\": 260, \"dropout_rate\": 0.0},\n        {\"hidden_neurons\": [80, 40], \"learning_rate\": 0.008, \"epochs\": 320, \"dropout_rate\": 0.0},\n        {\"hidden_neurons\": [96, 48, 24], \"learning_rate\": 0.006, \"epochs\": 400, \"dropout_rate\": 0.0},\n    ],\n}\n\n\ndef get_config():\n    \"\"\"Expose the experiment configuration for tests and docs.\"\"\"\n    return EXPERIMENT_CONFIG\n\n\ndef experiment_profile():\n    \"\"\"Return a compact summary of the assigned task and model design.\"\"\"\n    return build_experiment_profile(EXPERIMENT_CONFIG)\n\n\ndef validate_experiment_configuration():\n    \"\"\"Verify that the folder matches its assigned requirement.\"\"\"\n    return validate_config(EXPERIMENT_CONFIG)\n\n\nif __name__ == \"__main__\":\n    run_experiment(EXPERIMENT_CONFIG, Path(__file__).resolve().parent)\n",
  "experiment_support.py": "\"\"\"\nShared training utilities for the five experiment folders.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nimport random\nimport shutil\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import (\n    ConfusionMatrixDisplay,\n    classification_report,\n    confusion_matrix,\n    f1_score,\n)\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import OneHotEncoder\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\n\ntry:\n    from torchvision import datasets\nexcept Exception:  # pragma: no cover - only relevant for the MNIST experiment\n    datasets = None\n\n\nDEVICE = torch.device(\"cpu\")\nCOLUMN_NAMES = [\"buying\", \"maint\", \"doors\", \"persons\", \"lug_boot\", \"safety\", \"class\"]\nCAR_DATA_URL = \"https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data\"\nOUTPUT_DIR_NAME = \"outputs\"\nDATA_DIR_NAME = \"data\"\n\n\ndef expected_output_files():\n    \"\"\"List the files produced by a successful run.\"\"\"\n    return [\n        \"best_hyperparameters.json\",\n        \"best_model_learning_curve.png\",\n        \"classification_report.txt\",\n        \"dataset_split_summary.json\",\n        \"hyperparameter_search_results.csv\",\n        \"hyperparameter_validation_accuracy.png\",\n        \"test_confusion_matrix.png\",\n    ]\n\n\ndef build_experiment_profile(config):\n    \"\"\"Create a compact metadata summary for docs and tests.\"\"\"\n    return {\n        \"experiment_name\": config[\"experiment_name\"],\n        \"task_requirement\": config[\"task_requirement\"],\n        \"dataset_name\": config[\"dataset_name\"],\n        \"activation\": config[\"activation\"],\n        \"force_single_layer\": config[\"force_single_layer\"],\n        \"dropout_rates\": sorted({float(item[\"dropout_rate\"]) for item in config[\"hyperparameter_grid\"]}),\n        \"epoch_values\": sorted({int(item[\"epochs\"]) for item in config[\"hyperparameter_grid\"]}),\n        \"hidden_layer_counts\": sorted({len(item[\"hidden_neurons\"]) for item in config[\"hyperparameter_grid\"]}),\n    }\n\n\ndef validate_config(config):\n    \"\"\"Ensure each experiment matches its assigned requirement.\"\"\"\n    profile = build_experiment_profile(config)\n    task = profile[\"task_requirement\"]\n\n    if task == \"Add dropout in hidden layers\":\n        if profile[\"dataset_name\"] != \"car_evaluation\":\n            raise ValueError(\"The dropout experiment should use the Car Evaluation dataset.\")\n        if profile[\"activation\"] != \"relu\":\n            raise ValueError(\"The dropout experiment should keep ReLU as the activation.\")\n        if max(profile[\"dropout_rates\"]) <= 0.0:\n            raise ValueError(\"The dropout experiment must use positive dropout rates.\")\n        if min(profile[\"hidden_layer_counts\"]) < 1:\n            raise ValueError(\"The dropout experiment must keep at least one hidden layer.\")\n    elif task == \"Use tanh instead of ReLU\":\n        if profile[\"activation\"] != \"tanh\":\n            raise ValueError(\"The tanh experiment must use tanh activations.\")\n        if any(rate != 0.0 for rate in profile[\"dropout_rates\"]):\n            raise ValueError(\"The tanh experiment should not add dropout as its main change.\")\n    elif task == \"Use sigmoid instead of ReLU\":\n        if profile[\"activation\"] != \"sigmoid\":\n            raise ValueError(\"The sigmoid experiment must use sigmoid activations.\")\n        if any(rate != 0.0 for rate in profile[\"dropout_rates\"]):\n            raise ValueError(\"The sigmoid experiment should not add dropout as its main change.\")\n    elif task == \"Use a single layer network\":\n        if not profile[\"force_single_layer\"]:\n            raise ValueError(\"The single-layer experiment must mark itself as single-layer.\")\n        if max(profile[\"hidden_layer_counts\"]) != 0:\n            raise ValueError(\"The single-layer experiment must not define hidden layers.\")\n    elif task == \"Use only 5 epochs with MNIST dataset\":\n        if profile[\"dataset_name\"] != \"mnist\":\n            raise ValueError(\"The MNIST experiment must use the MNIST dataset.\")\n        if profile[\"epoch_values\"] != [5]:\n            raise ValueError(\"Every MNIST configuration must run for exactly 5 epochs.\")\n    else:\n        raise ValueError(f\"Unknown task requirement: {task}\")\n\n    return True\n\n\ndef ensure_directory(path: Path):\n    \"\"\"Create a directory and all its parents when needed.\"\"\"\n    path.mkdir(parents=True, exist_ok=True)\n\n\ndef set_reproducibility(seed: int):\n    \"\"\"Seed the random generators used during training.\"\"\"\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef get_activation_module(name: str):\n    \"\"\"Map a configured activation name to a PyTorch activation layer.\"\"\"\n    activation_map = {\n        \"relu\": nn.ReLU,\n        \"tanh\": nn.Tanh,\n        \"sigmoid\": nn.Sigmoid,\n    }\n    if name not in activation_map:\n        raise ValueError(f\"Unsupported activation: {name}\")\n    return activation_map[name]()\n\n\nclass ConfigurableMLP(nn.Module):\n    \"\"\"Multi-layer perceptron whose hidden layout is defined by the config.\"\"\"\n\n    def __init__(self, input_dim, num_classes, hidden_neurons, activation_name, dropout_rate):\n        super().__init__()\n        layers = []\n        in_features = input_dim\n\n        for hidden_units in hidden_neurons:\n            layers.append(nn.Linear(in_features, hidden_units))\n            layers.append(get_activation_module(activation_name))\n            if dropout_rate > 0:\n                layers.append(nn.Dropout(dropout_rate))\n            in_features = hidden_units\n\n        layers.append(nn.Linear(in_features, num_classes))\n        self.network = nn.Sequential(*layers)\n\n    def forward(self, inputs):\n        return self.network(inputs)\n\n\ndef load_car_dataframe(data_dir: Path):\n    \"\"\"Load the Car Evaluation dataset from a local cache or from the UCI source.\"\"\"\n    ensure_directory(data_dir)\n    cached_path = data_dir / \"car.data\"\n    if cached_path.exists():\n        return pd.read_csv(cached_path, header=None, names=COLUMN_NAMES)\n\n    dataframe = pd.read_csv(CAR_DATA_URL, header=None, names=COLUMN_NAMES)\n    dataframe.to_csv(cached_path, header=False, index=False)\n    return dataframe\n\n\ndef prepare_car_dataset_bundle(config, base_dir: Path):\n    \"\"\"Encode and split the Car Evaluation dataset.\"\"\"\n    dataframe = load_car_dataframe(base_dir / DATA_DIR_NAME)\n    x_raw = dataframe.drop(columns=[\"class\"])\n    y_raw = dataframe[\"class\"]\n\n    encoder = OneHotEncoder(sparse_output=False, handle_unknown=\"ignore\")\n    x_encoded = encoder.fit_transform(x_raw).astype(np.float32)\n\n    class_names = sorted(y_raw.unique().tolist())\n    class_to_index = {name: index for index, name in enumerate(class_names)}\n    y_encoded = y_raw.map(class_to_index).to_numpy(dtype=np.int64)\n\n    x_train_val, x_test, y_train_val, y_test = train_test_split(\n        x_encoded,\n        y_encoded,\n        test_size=config[\"test_size\"],\n        random_state=config[\"random_seed\"],\n        stratify=y_encoded,\n    )\n    x_train, x_val, y_train, y_val = train_test_split(\n        x_train_val,\n        y_train_val,\n        test_size=config[\"validation_size\"],\n        random_state=config[\"random_seed\"],\n        stratify=y_train_val,\n    )\n\n    return {\n        \"x_train\": x_train,\n        \"x_val\": x_val,\n        \"x_test\": x_test,\n        \"y_train\": y_train,\n        \"y_val\": y_val,\n        \"y_test\": y_test,\n        \"class_names\": class_names,\n        \"input_dim\": int(x_encoded.shape[1]),\n        \"num_classes\": len(class_names),\n        \"split_strategy\": \"stratified_random_split\",\n    }\n\n\ndef prepare_mnist_dataset_bundle(config, base_dir: Path):\n    \"\"\"Load the MNIST dataset, flatten the images, and create a validation split.\"\"\"\n    if datasets is None:\n        raise ImportError(\"torchvision is required for the MNIST experiment.\")\n\n    data_dir = base_dir / DATA_DIR_NAME\n    ensure_directory(data_dir)\n\n    train_dataset = datasets.MNIST(root=data_dir, train=True, download=True)\n    test_dataset = datasets.MNIST(root=data_dir, train=False, download=True)\n\n    x_full = train_dataset.data.numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0\n    y_full = train_dataset.targets.numpy().astype(np.int64)\n    x_test = test_dataset.data.numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0\n    y_test = test_dataset.targets.numpy().astype(np.int64)\n\n    x_train, x_val, y_train, y_val = train_test_split(\n        x_full,\n        y_full,\n        test_size=config[\"validation_size\"],\n        random_state=config[\"random_seed\"],\n        stratify=y_full,\n    )\n\n    return {\n        \"x_train\": x_train,\n        \"x_val\": x_val,\n        \"x_test\": x_test,\n        \"y_train\": y_train,\n        \"y_val\": y_val,\n        \"y_test\": y_test,\n        \"class_names\": [str(index) for index in range(10)],\n        \"input_dim\": 28 * 28,\n        \"num_classes\": 10,\n        \"split_strategy\": \"official_test_set_with_stratified_validation_split\",\n    }\n\n\ndef prepare_dataset_bundle(config, base_dir: Path):\n    \"\"\"Dispatch to the correct dataset loader.\"\"\"\n    if config[\"dataset_name\"] == \"car_evaluation\":\n        return prepare_car_dataset_bundle(config, base_dir)\n    if config[\"dataset_name\"] == \"mnist\":\n        return prepare_mnist_dataset_bundle(config, base_dir)\n    raise ValueError(f\"Unsupported dataset: {config['dataset_name']}\")\n\n\ndef compute_class_weights(y_train, number_of_classes):\n    \"\"\"Use inverse-frequency class weights for balanced optimization.\"\"\"\n    counts = np.bincount(y_train, minlength=number_of_classes).astype(np.float32)\n    weights = len(y_train) / (number_of_classes * counts)\n    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)\n\n\ndef create_data_loader(x_values, y_values, batch_size):\n    \"\"\"Build a shuffled DataLoader for mini-batch optimization.\"\"\"\n    features = torch.tensor(x_values, dtype=torch.float32)\n    labels = torch.tensor(y_values, dtype=torch.long)\n    dataset = TensorDataset(features, labels)\n    return DataLoader(dataset, batch_size=batch_size, shuffle=True)\n\n\ndef evaluate_model(model, x_values, y_values):\n    \"\"\"Return accuracy, macro F1, and predictions for one split.\"\"\"\n    model.eval()\n    with torch.no_grad():\n        logits = model(torch.tensor(x_values, dtype=torch.float32, device=DEVICE))\n        predictions = torch.argmax(logits, dim=1).cpu().numpy()\n\n    accuracy = float(np.mean(predictions == y_values))\n    macro_f1 = float(f1_score(y_values, predictions, average=\"macro\", zero_division=0))\n    return accuracy, macro_f1, predictions\n\n\ndef compute_split_loss(model, criterion, x_values, y_values):\n    \"\"\"Measure the weighted cross-entropy loss for reporting.\"\"\"\n    model.eval()\n    with torch.no_grad():\n        inputs = torch.tensor(x_values, dtype=torch.float32, device=DEVICE)\n        labels = torch.tensor(y_values, dtype=torch.long, device=DEVICE)\n        logits = model(inputs)\n        return float(criterion(logits, labels).item())\n\n\ndef train_one_model(dataset_bundle, config, search_config):\n    \"\"\"Train one hyperparameter configuration and collect its history.\"\"\"\n    model = ConfigurableMLP(\n        input_dim=dataset_bundle[\"input_dim\"],\n        num_classes=dataset_bundle[\"num_classes\"],\n        hidden_neurons=search_config[\"hidden_neurons\"],\n        activation_name=config[\"activation\"],\n        dropout_rate=search_config[\"dropout_rate\"],\n    ).to(DEVICE)\n\n    class_weights = compute_class_weights(dataset_bundle[\"y_train\"], dataset_bundle[\"num_classes\"])\n    criterion = nn.CrossEntropyLoss(weight=class_weights)\n    optimizer = torch.optim.Adam(model.parameters(), lr=search_config[\"learning_rate\"])\n    loader = create_data_loader(dataset_bundle[\"x_train\"], dataset_bundle[\"y_train\"], config[\"batch_size\"])\n    history = {\"train_loss\": [], \"train_accuracy\": [], \"val_accuracy\": [], \"val_macro_f1\": []}\n\n    for _ in range(search_config[\"epochs\"]):\n        model.train()\n        for features, labels in loader:\n            features = features.to(DEVICE)\n            labels = labels.to(DEVICE)\n            optimizer.zero_grad()\n            logits = model(features)\n            loss = criterion(logits, labels)\n            loss.backward()\n            optimizer.step()\n\n        train_loss = compute_split_loss(model, criterion, dataset_bundle[\"x_train\"], dataset_bundle[\"y_train\"])\n        train_accuracy, _, _ = evaluate_model(model, dataset_bundle[\"x_train\"], dataset_bundle[\"y_train\"])\n        val_accuracy, val_macro_f1, _ = evaluate_model(model, dataset_bundle[\"x_val\"], dataset_bundle[\"y_val\"])\n\n        history[\"train_loss\"].append(train_loss)\n        history[\"train_accuracy\"].append(train_accuracy)\n        history[\"val_accuracy\"].append(val_accuracy)\n        history[\"val_macro_f1\"].append(val_macro_f1)\n\n    return {\n        \"model\": model,\n        \"history\": history,\n        \"best_val_accuracy\": max(history[\"val_accuracy\"]),\n        \"best_val_macro_f1\": max(history[\"val_macro_f1\"]),\n        \"final_val_accuracy\": history[\"val_accuracy\"][-1],\n        \"final_val_macro_f1\": history[\"val_macro_f1\"][-1],\n    }\n\n\ndef run_hyperparameter_search(dataset_bundle, config):\n    \"\"\"Select the best configuration according to validation macro F1.\"\"\"\n    search_rows = []\n    best_result = None\n    best_config = None\n\n    for trial_number, search_config in enumerate(config[\"hyperparameter_grid\"], start=1):\n        print(f\"Trial {trial_number}: {search_config}\")\n        set_reproducibility(config[\"random_seed\"] + trial_number)\n        result = train_one_model(dataset_bundle, config, search_config)\n\n        row = {\n            \"trial\": trial_number,\n            \"hidden_neurons\": str(search_config[\"hidden_neurons\"]),\n            \"learning_rate\": float(search_config[\"learning_rate\"]),\n            \"epochs\": int(search_config[\"epochs\"]),\n            \"dropout_rate\": float(search_config[\"dropout_rate\"]),\n            \"activation\": config[\"activation\"],\n            \"best_val_accuracy\": float(result[\"best_val_accuracy\"]),\n            \"best_val_macro_f1\": float(result[\"best_val_macro_f1\"]),\n            \"final_val_accuracy\": float(result[\"final_val_accuracy\"]),\n            \"final_val_macro_f1\": float(result[\"final_val_macro_f1\"]),\n        }\n        search_rows.append(row)\n\n        if best_result is None or result[\"best_val_macro_f1\"] > best_result[\"best_val_macro_f1\"]:\n            best_result = result\n            best_config = search_config\n\n    return best_config, best_result, pd.DataFrame(search_rows)\n\n\ndef save_plots(output_dir: Path, search_table, best_history, y_test, y_pred, class_names):\n    \"\"\"Export the figures used by the notebook and the report.\"\"\"\n    ensure_directory(output_dir)\n\n    label_values = [\n        f\"{row.hidden_neurons}\\nlr={row.learning_rate}\\ne={row.epochs}\\ndrop={row.dropout_rate}\"\n        for row in search_table.itertuples(index=False)\n    ]\n\n    plt.figure(figsize=(11, 5))\n    plt.bar(label_values, search_table[\"best_val_macro_f1\"])\n    plt.ylim(0, 1)\n    plt.ylabel(\"Best validation macro F1\")\n    plt.title(\"Hyperparameter search results\")\n    plt.tight_layout()\n    plt.savefig(output_dir / \"hyperparameter_validation_accuracy.png\", dpi=160)\n    plt.close()\n\n    epochs = np.arange(1, len(best_history[\"train_loss\"]) + 1)\n    plt.figure(figsize=(10, 5))\n    plt.plot(epochs, best_history[\"train_accuracy\"], label=\"Train accuracy\")\n    plt.plot(epochs, best_history[\"val_accuracy\"], label=\"Validation accuracy\")\n    plt.plot(epochs, best_history[\"val_macro_f1\"], label=\"Validation macro F1\")\n    plt.xlabel(\"Epoch\")\n    plt.ylabel(\"Score\")\n    plt.title(\"Best model learning curve\")\n    plt.legend()\n    plt.tight_layout()\n    plt.savefig(output_dir / \"best_model_learning_curve.png\", dpi=160)\n    plt.close()\n\n    matrix = confusion_matrix(y_test, y_pred)\n    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=class_names)\n    figure, axis = plt.subplots(figsize=(8, 7))\n    display.plot(ax=axis, cmap=\"Blues\", colorbar=False)\n    plt.title(\"Test confusion matrix\")\n    plt.tight_layout()\n    plt.savefig(output_dir / \"test_confusion_matrix.png\", dpi=160)\n    plt.close(figure)\n\n\ndef write_summary_files(output_dir: Path, dataset_bundle, config, best_config, best_result, search_table, report):\n    \"\"\"Save the text and metadata files used by the report.\"\"\"\n    ensure_directory(output_dir)\n\n    search_table.to_csv(output_dir / \"hyperparameter_search_results.csv\", index=False)\n\n    with open(output_dir / \"classification_report.txt\", \"w\", encoding=\"utf-8\") as file:\n        file.write(report)\n\n    with open(output_dir / \"best_hyperparameters.json\", \"w\", encoding=\"utf-8\") as file:\n        json.dump(\n            {\n                \"task_requirement\": config[\"task_requirement\"],\n                \"dataset_name\": config[\"dataset_name\"],\n                \"activation\": config[\"activation\"],\n                \"dropout_rate\": float(best_config[\"dropout_rate\"]),\n                \"hidden_neurons\": best_config[\"hidden_neurons\"],\n                \"learning_rate\": float(best_config[\"learning_rate\"]),\n                \"epochs\": int(best_config[\"epochs\"]),\n            },\n            file,\n            indent=2,\n        )\n\n    with open(output_dir / \"dataset_split_summary.json\", \"w\", encoding=\"utf-8\") as file:\n        json.dump(\n            {\n                \"experiment_name\": config[\"experiment_name\"],\n                \"task_requirement\": config[\"task_requirement\"],\n                \"dataset_name\": config[\"dataset_name\"],\n                \"random_seed\": config[\"random_seed\"],\n                \"test_size\": config[\"test_size\"],\n                \"validation_size_inside_train_val\": config[\"validation_size\"],\n                \"train_rows\": int(len(dataset_bundle[\"x_train\"])),\n                \"validation_rows\": int(len(dataset_bundle[\"x_val\"])),\n                \"test_rows\": int(len(dataset_bundle[\"x_test\"])),\n                \"input_features\": int(dataset_bundle[\"input_dim\"]),\n                \"class_names\": dataset_bundle[\"class_names\"],\n                \"split_strategy\": dataset_bundle[\"split_strategy\"],\n                \"best_validation_macro_f1\": float(best_result[\"best_val_macro_f1\"]),\n            },\n            file,\n            indent=2,\n        )\n\n\ndef zip_outputs(base_dir: Path, experiment_name: str):\n    \"\"\"Create a zip archive of the generated files inside the outputs directory.\"\"\"\n    archive_base = base_dir / f\"{experiment_name}_results\"\n    archive_path = base_dir / f\"{experiment_name}_results.zip\"\n    if archive_path.exists():\n        archive_path.unlink()\n    shutil.make_archive(str(archive_base), \"zip\", base_dir / OUTPUT_DIR_NAME)\n    return archive_path.name\n\n\ndef run_experiment(config, base_dir: Path):\n    \"\"\"Run the end-to-end experiment for one folder.\"\"\"\n    validate_config(config)\n    set_reproducibility(config[\"random_seed\"])\n\n    output_dir = base_dir / OUTPUT_DIR_NAME\n    ensure_directory(output_dir)\n\n    dataset_bundle = prepare_dataset_bundle(config, base_dir)\n    best_config, best_result, search_table = run_hyperparameter_search(dataset_bundle, config)\n    test_accuracy, test_macro_f1, y_pred = evaluate_model(\n        best_result[\"model\"],\n        dataset_bundle[\"x_test\"],\n        dataset_bundle[\"y_test\"],\n    )\n\n    report = classification_report(\n        dataset_bundle[\"y_test\"],\n        y_pred,\n        target_names=dataset_bundle[\"class_names\"],\n        digits=4,\n        zero_division=0,\n    )\n\n    print(\"\\nTask requirement:\")\n    print(config[\"task_requirement\"])\n    print(\"\\nBest hyperparameters:\")\n    print(best_config)\n    print(f\"\\nTest accuracy: {test_accuracy:.4f}\")\n    print(f\"Test macro F1: {test_macro_f1:.4f}\")\n    print(\"\\nClassification report:\")\n    print(report)\n\n    write_summary_files(output_dir, dataset_bundle, config, best_config, best_result, search_table, report)\n    save_plots(\n        output_dir,\n        search_table,\n        best_result[\"history\"],\n        dataset_bundle[\"y_test\"],\n        y_pred,\n        dataset_bundle[\"class_names\"],\n    )\n    archive_name = zip_outputs(base_dir, config[\"experiment_name\"])\n    print(f\"\\nSaved outputs in: {output_dir}\")\n    print(f\"Created zip archive: {archive_name}\")\n\n",
  "tests/__init__.py": "# Package marker for pytest collection.\n",
  "tests/helpers.py": "import importlib.util\nfrom pathlib import Path\n\n\ndef get_module():\n    module_path = Path(__file__).resolve().parents[1] / \"run_car_mlp.py\"\n    spec = importlib.util.spec_from_file_location(\"experiment_3_module\", module_path)\n    module = importlib.util.module_from_spec(spec)\n    assert spec.loader is not None\n    spec.loader.exec_module(module)\n    return module\n",
  "tests/test_sigmoid_config.py": "import pytest\n\nfrom .helpers import get_module\n\n\n@pytest.mark.parametrize(\n    (\"field\", \"expected\"),\n    [\n        (\"task_requirement\", \"Use sigmoid instead of ReLU\"),\n        (\"dataset_name\", \"car_evaluation\"),\n        (\"activation\", \"sigmoid\"),\n    ],\n)\ndef test_core_configuration_fields(field, expected):\n    module = get_module()\n    assert module.get_config()[field] == expected\n\n\ndef test_module_validation_passes():\n    module = get_module()\n    assert module.validate_experiment_configuration() is True\n",
  "tests/test_sigmoid_grid.py": "from .helpers import get_module\n\n\ndef test_search_grid_uses_distinct_learning_rates_and_epochs():\n    module = get_module()\n    grid = module.get_config()[\"hyperparameter_grid\"]\n    assert len({config[\"learning_rate\"] for config in grid}) == 4\n    assert len({config[\"epochs\"] for config in grid}) == 4\n\n\ndef test_search_grid_disables_dropout():\n    module = get_module()\n    assert all(config[\"dropout_rate\"] == 0.0 for config in module.get_config()[\"hyperparameter_grid\"])\n\n\ndef test_artifact_manifest_is_available():\n    module = get_module()\n    outputs = module.expected_output_files()\n    assert \"classification_report.txt\" in outputs\n"
}

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
for relative_path, content in FILES.items():
    target = EXPERIMENT_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

try:
    import pytest  # noqa: F401
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pytest"])

print(f"Prepared {len(FILES)} files in {EXPERIMENT_DIR}")
%cd /content/experiment_3_seed99


In [ ]:
import sys
import torch
import pytest
print(sys.version)
print("torch", torch.__version__)
print("pytest", pytest.__version__)


In [ ]:
%run run_car_mlp.py


In [ ]:
!pytest tests -q


In [ ]:
from IPython.display import Image, display
display(Image('outputs/hyperparameter_validation_accuracy.png'))
display(Image('outputs/best_model_learning_curve.png'))
display(Image('outputs/test_confusion_matrix.png'))
